# Prüfungsleistung - Strukturierte Informationsextraktion aus Stellenanzeigen

**Name:** *Jakob Wilmesmeier*

---

## Kontext: Die JobMatch-Plattform

Die (fiktive) HR-Tech-Firma **JobMatch** betreibt eine Karriereplattform, auf der Bewerber:innen passende Stellenanzeigen finden. Täglich werden tausende Anzeigen aus dem Web gecrawlt, aber als unstrukturierter Volltext sind sie für die Nutzer:innen kaum filterbar.

JobMatch braucht ein Modell, das aus jeder Anzeige genau die Felder extrahiert, nach denen Bewerber:innen tatsächlich filtern: Jobtitel, Unternehmen, Erfahrungslevel, Skills, Bildungsanforderungen, Gehaltsspanne und Benefits.

## Aufgabe

Entwickle ein Modell, das aus einer englischsprachigen Stellenanzeige (Freitext) ein strukturiertes JSON-Objekt mit sieben vordefinierten Feldern extrahiert:

```json
{
  "job_title":              "Senior Data Engineer",
  "company":                "Acme Corp",
  "experience_level":       "senior",
  "required_skills":        ["Python", "SQL", "Airflow"],
  "education_requirements": "Bachelor's degree in Computer Science",
  "salary_range":           {"min": 80000, "max": 110000, "currency": "USD"},
  "compensation_benefits":  ["health insurance", "401k matching", "remote work"]
}
```

Feldtypen im Detail:
- `job_title` (string): Bezeichnung der Stelle
- `company` (string): Name des Unternehmens
- `experience_level` (enum): Einer von `entry`, `junior`, `mid`, `senior`, `lead`, `executive`, `unknown`
- `required_skills` (list[string]): Geforderte technische oder fachliche Skills
- `education_requirements` (string): Bildungsanforderungen (oder leerer String)
- `salary_range` (object): `{min, max, currency}`, jeweils `null` wenn nicht angegeben
- `compensation_benefits` (list[string]): Benefits (Versicherungen, Urlaub, Remote etc.)

Felder können leer sein, wenn die Anzeige keine entsprechende Information enthält (leerer String, leere Liste, `null`-Subfelder bei `salary_range`).

**Inferenz-Laufzeit**: Das Ausführen des Auswertungs-Abschnitts (Modell laden + `predict()` auf 50 Testanzeigen + `evaluate()`) darf insgesamt maximal 20 Minuten dauern, gemessen auf einer Colab-T4-GPU. Plane dein Modell entsprechend. Bei Überschreitung gibt es Abzug in der manuellen Bewertung (siehe weiter unten).

## Datensatz

Du erhältst eine ZIP-Datei `train_data.zip` mit Trainingsdaten: Paare aus `.txt`-Anzeige und `.json`-Label im obigen Schema. Auf den Testdaten (die du nicht siehst) wird dein Modell am Ende automatisch bewertet.

## Abgabe

Du gibst eine **ZIP-Datei** ab, die folgendes enthält:

- `notebook.ipynb` – dieses Notebook, mit deinen Implementierungen
- `model/` – ein Ordner mit deinem trainierten Modell (Tokenizer, Modellgewichte, ggf. weitere Artefakte wie Mapping-Dictionaries)

Ich werde **nur die Zellen bis einschließlich der Auswertung** ausführen. Dein Trainings-Code wird **nicht** erneut ausgeführt, sondern nur gesichtet. Daher ist es sehr wichtig, dass dein Modell in der `predict()`-Funktion korrekt aus dem `model/`-Ordner geladen wird.

## Bewertung

Insgesamt **30 Punkte**:

### 10 Punkte automatisch

Berechnet über `evaluate()` als Macro-F1 über alle 7 Felder.
- Macro-F1 < 0.3 → 0 Punkte
- Macro-F1 ≥ 0.85 → 10 Punkte
- dazwischen linear

### 20 Punkte manuell

In fünf Bereichen à 4 Punkten:

| Bereich | Was wird bewertet? |
|---|---|
| **Methoden- und Modellwahl** | Passt das Modell zur Aufgabe? Begründet? Alternativen erwogen? |
| **Data Understanding & Preprocessing** | Sinnvoll exploriert? Preprocessing zur Modellwahl passend? Sauberer Split? |
| **Trainings- und Inferenz-Pipeline** | Reproduzierbar? Hyperparameter begründet? Code lesbar? |
| **Evaluation & Fehleranalyse** | Eigene Evaluation jenseits von `evaluate()`? F1 pro Feld? Schwächen erkannt? |
| **Reflexion & Begründung** | Werden Entscheidungen in Markdown-Zellen erklärt? Wissenschaftlicher Stil? |
---

## Vorbereitungen

Lasse die folgenden Zellen unverändert. Sie installieren benötigte Pakete, definieren das Zielschema und entpacken die Trainingsdaten.

In [ ]:
from enum import StrEnum
from typing import Optional
from pydantic import BaseModel, Field


class ExperienceLevel(StrEnum):
    ENTRY = "entry"
    JUNIOR = "junior"
    MID = "mid"
    SENIOR = "senior"
    LEAD = "lead"
    EXECUTIVE = "executive"
    UNKNOWN = "unknown"


class SalaryRange(BaseModel):
    min: Optional[int] = None
    max: Optional[int] = None
    currency: Optional[str] = None


class JobPosting(BaseModel):
    job_title: str = ""
    company: str = ""
    experience_level: ExperienceLevel = ExperienceLevel.UNKNOWN
    required_skills: list[str] = Field(default_factory=list)
    education_requirements: str = ""
    salary_range: SalaryRange = Field(default_factory=SalaryRange)
    compensation_benefits: list[str] = Field(default_factory=list)

In [ ]:
import zipfile
from pathlib import Path

TRAIN_ZIP = Path("train_data.zip")
TRAIN_DIR = Path("train")

if not TRAIN_DIR.exists():
    with zipfile.ZipFile(TRAIN_ZIP) as zf:
        zf.extractall(".")
print(f"Train: {len(list(TRAIN_DIR.glob('*.txt')))} Anzeigen")

Train: 700 Anzeigen


## Modell laden und Inferenz

In diesem Abschnitt wird dein **trainiertes Modell aus dem `model/`-Ordner geladen** und die `predict()`-Funktion bereitgestellt. Dieser Abschnitt wird bei der Korrektur ausgeführt – die Trainings-Sektion (weiter unten) hingegen nicht.

Passe die folgenden Zellen so an, dass:
1. dein Modell und alle benötigten Artefakte aus dem lokalen `model/`-Ordner geladen werden,
2. die `predict()`-Funktion ein gültiges `JobPosting`-Objekt zurückgibt.

**Verändere die Signatur und den Rückgabetyp von `predict()` nicht.**

In [ ]:
# Lade hier dein trainiertes Modell und alle benötigten Artefakte aus dem `model/`-Ordner.
# Beispiel für ein Hugging-Face-Modell:
#
#   from transformers import AutoTokenizer, AutoModelForTokenClassification
#   MODEL_DIR = Path("model")
#   tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
#   model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

# ...


In [ ]:
def predict(text: str) -> JobPosting:
    # ...

    return JobPosting(
        job_title="",
        company="",
        experience_level=ExperienceLevel.UNKNOWN,
        required_skills=[],
        education_requirements="",
        salary_range=SalaryRange(),
        compensation_benefits=[],
    )

## Auswertung

**Lasse die `evaluate()`-Funktion unverändert.** Sie bewertet dein Modell und berechnet die automatischen Punkte (max. 10).

Während der Entwicklung kannst du `EVAL_DIR` in der nächsten Zelle z.B. auf `Path("train")` setzen, um die Evaluation auf den Trainingsdaten laufen zu lassen und ein erstes Gefühl für die Modellgüte zu bekommen. Die Performance auf den Trainingsdaten überschätzt allerdings, wie gut dein Modell auf neuen Daten generalisiert, ein eigener Train/Val-Split ist methodisch sauberer.

In [ ]:
import json
import math
import time
from pathlib import Path


def _norm_text(s: str) -> str:
    """Normalisiert Strings für robusten Vergleich (lowercase, getrimmt, Whitespace zusammengezogen)."""
    return " ".join((s or "").lower().split())


def _f1(tp: int, fp: int, fn: int) -> float:
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)


def _score_string_field(pred: str, true: str) -> tuple[int, int, int]:
    """Für String-Felder: token-set-basierter F1. Liefert (tp, fp, fn)."""
    p_tokens = set(_norm_text(pred).split())
    t_tokens = set(_norm_text(true).split())
    if not p_tokens and not t_tokens:
        return (1, 0, 0)  # beide leer -> perfekter Match
    tp = len(p_tokens & t_tokens)
    fp = len(p_tokens - t_tokens)
    fn = len(t_tokens - p_tokens)
    return (tp, fp, fn)


def _score_enum_field(pred: str, true: str) -> tuple[int, int, int]:
    """Für Enum-Felder: exakter Match."""
    if pred == true:
        return (1, 0, 0)
    return (0, 1, 1)


def _score_list_field(pred: list[str], true: list[str]) -> tuple[int, int, int]:
    """Für Listen-Felder: Set-F1 mit normalisierten Items."""
    p_set = {_norm_text(x) for x in pred if x}
    t_set = {_norm_text(x) for x in true if x}
    if not p_set and not t_set:
        return (1, 0, 0)
    tp = len(p_set & t_set)
    fp = len(p_set - t_set)
    fn = len(t_set - p_set)
    return (tp, fp, fn)


def _score_salary_field(pred: dict, true: dict) -> tuple[int, int, int]:
    """Für salary_range: F1 über die drei Subfelder. Toleranz für min/max: 10%."""
    tp = fp = fn = 0
    for key in ("min", "max"):
        p_val, t_val = pred.get(key), true.get(key)
        if p_val is None and t_val is None:
            tp += 1
        elif p_val is None and t_val is not None:
            fn += 1
        elif p_val is not None and t_val is None:
            fp += 1
        else:
            # 10% Toleranz
            if abs(p_val - t_val) <= max(1000, 0.1 * t_val):
                tp += 1
            else:
                fp += 1
                fn += 1
    # Currency: exakt
    p_cur, t_cur = pred.get("currency"), true.get("currency")
    if p_cur is None and t_cur is None:
        tp += 1
    elif (p_cur or "").upper() == (t_cur or "").upper() and p_cur:
        tp += 1
    else:
        if p_cur is not None:
            fp += 1
        if t_cur is not None:
            fn += 1
    return (tp, fp, fn)


FIELD_SCORERS = {
    "job_title": _score_string_field,
    "company": _score_string_field,
    "experience_level": _score_enum_field,
    "required_skills": _score_list_field,
    "education_requirements": _score_string_field,
    "salary_range": _score_salary_field,
    "compensation_benefits": _score_list_field,
}

INFERENCE_TIME_LIMIT_SECONDS = 20 * 60  # 20 Minuten


def evaluate(test_folder: Path) -> int:
    """Bewertet das Modell auf dem Testset und gibt die automatischen Punkte (0-10) zurück."""
    test_folder = Path(test_folder)
    txt_files = sorted(test_folder.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"Keine .txt-Dateien in {test_folder} gefunden.")

    field_counts = {f: [0, 0, 0] for f in FIELD_SCORERS}

    start_time = time.time()
    for txt_file in txt_files:
        json_file = txt_file.with_suffix(".json")
        if not json_file.exists():
            print(f"WARNUNG: Kein Label für {txt_file.name}, übersprungen.")
            continue

        text = txt_file.read_text(encoding="utf-8")
        true = json.loads(json_file.read_text(encoding="utf-8"))
        pred = predict(text).model_dump()

        for field, scorer in FIELD_SCORERS.items():
            tp, fp, fn = scorer(pred[field], true[field])
            field_counts[field][0] += tp
            field_counts[field][1] += fp
            field_counts[field][2] += fn
    inference_time = time.time() - start_time

    print("\n=== F1 pro Feld ===")
    f1_per_field = {}
    for field, (tp, fp, fn) in field_counts.items():
        f1 = _f1(tp, fp, fn)
        f1_per_field[field] = f1
        print(f"  {field:25s}  F1={f1:.3f}  (TP={tp}, FP={fp}, FN={fn})")

    macro_f1 = sum(f1_per_field.values()) / len(f1_per_field)
    print(f"\nMacro-F1: {macro_f1:.3f}")

    # Punkteberechnung: <0.3 -> 0 Punkte, >=0.85 -> 10 Punkte, dazwischen linear
    if macro_f1 < 0.3:
        points = 0
    else:
        points = min(10, math.ceil((macro_f1 - 0.3) * 18))
    print(f"Automatische Punkte: {points} / 10")

    # Inferenz-Laufzeit
    print(f"\nInferenzzeit: {inference_time:.1f}s "
          f"({inference_time/len(txt_files):.2f}s pro Anzeige)")
    if inference_time > INFERENCE_TIME_LIMIT_SECONDS:
        print(f"WARNUNG: Inferenz hat das 20-Minuten-Limit überschritten "
              f"({inference_time/60:.1f} Min).")

    return points

In [ ]:
# TEST_ZIP = Path("test_data.zip")
# TEST_DIR = Path("test")
# if not TEST_DIR.exists() and TEST_ZIP.exists():
#     with zipfile.ZipFile(TEST_ZIP) as zf:
#         zf.extractall(".")
# print(f"Test: {len(list(TEST_DIR.glob('*.txt')))} Anzeigen")

In [ ]:
# Während der Entwicklung kannst du EVAL_DIR z.B. auf Path("train") setzen,
# um die Evaluation auf den Trainingsdaten auszuprobieren.
# Für die offizielle Bewertung wird hier Path("test") gesetzt.
EVAL_DIR = Path("test")

evaluate(EVAL_DIR)

---

# Trainings-Dokumentation

**Hinweis:** Der folgende Abschnitt dokumentiert dein Vorgehen beim Training. Er wird bei der Korrektur **nicht** ausgeführt, sondern nur gesichtet (das fertige Modell liegt ja bereits im `model/`-Ordner und wird oben geladen).

Dieser Abschnitt fließt in die **manuelle Bewertung** ein. Achte auf:
- Nachvollziehbare Darstellung deiner Entscheidungen (in Markdown-Zellen)
- Reproduzierbarer Code (falls jemand das Training nachvollziehen will)
- Sinnvolle Visualisierungen und Statistiken

## Data Understanding

Untersuche die bereitgestellten Daten. Sinnvolle Fragen können sein:
- Wie lang sind die Anzeigen typischerweise (in Tokens / Zeichen)?
- Wie ist die Verteilung der `experience_level`-Klassen?
- Wie viele Anzeigen enthalten überhaupt eine Gehaltsangabe?
- Welche Skills tauchen am häufigsten auf?

Diese Einsichten helfen dir, das richtige Modell und passende Preprocessing-Schritte zu wählen.

In [ ]:
# ...

## Data Preprocessing

Bereite die Daten so vor, dass sie zu deinem gewählten Modellansatz passen.
Mögliche Schritte: Tokenisierung, Konvertierung in Seq2Seq-Format (z.B. `text -> JSON-String`), Erzeugung von BIO-Tags für NER, Train/Val-Split, etc.

In [ ]:
# ...

## Modelldefinition und Training

Definiere und trainiere dein Modell. Begründe in einer Markdown-Zelle kurz, warum du dich für diesen Ansatz entschieden hast.

**Wichtig:** Speichere dein trainiertes Modell am Ende dieses Abschnitts in den Ordner `model/`, damit die `predict()`-Funktion ganz oben es laden kann.

Beispiel für ein Hugging-Face-Modell:
```python
model.save_pretrained("model")
tokenizer.save_pretrained("model")
```

In [ ]:
# ...


## Evaluation und Fehleranalyse

Über die `evaluate()`-Funktion hinaus: Wo macht dein Modell typische Fehler? Welche Felder sind am schwierigsten zu extrahieren und warum? Was wären sinnvolle nächste Schritte, um die Performance zu verbessern?

Erstelle hier eine kurze Fehleranalyse (z.B. F1 pro Feld auf einem Validierungs-Split, Beispiele für Fehler, Verwechslungsmatrix für `experience_level`).

In [ ]:
# ...
